In [2]:
print("OK")

OK


In [7]:
#setup
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve))

from dotenv import load_dotenv
load_dotenv()

print("setup complete")

#installing dependencies

setup complete


In [14]:
#testing config

import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "no Set OPENAI_API_KEY in .env file"
print(f" yes API Key loaded: {OPENAI_API_KEY[:8]}...")

PDF_PATH = "/Users/nihal/Downloads/Agentic chatbot/agentic-rag-chatbot-Nihal7778/sample_docs/Arxiv.pdf"
assert os.path.exists(PDF_PATH), f"no PDF not found at {PDF_PATH}"
print(f"yes PDF found: {PDF_PATH}")



 yes API Key loaded: sk-proj-...
yes PDF found: /Users/nihal/Downloads/Agentic chatbot/agentic-rag-chatbot-Nihal7778/sample_docs/Arxiv.pdf


In [16]:
#extracting text from pdf
import fitz # PyMuPDF
doc = fitz.open(PDF_PATH)
full_text = ""
page_texts = []

for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text")
    page_texts.append({"page": page_num + 1, "text": text, "start": len(full_text)})
    full_text += text + "\n"

doc.close()

print(f" Pages: {len(page_texts)}")
print(f" Total chars: {len(full_text)}")
print(f"\n--- First 500 chars ---\n{full_text[:500]}")

 Pages: 154
 Total chars: 368487

--- First 500 chars ---
IntechOpen Series  
Artificial Intelligence, Volume 7
Machine Learning 
Algorithms, Models and Applications
Edited by Jaydip Sen


Machine Learning - 
Algorithms, Models and 
Applications
Edited by Jaydip Sen
Published in London, United Kingdom


Supporting open minds since 2005

Machine Learning - Algorithms, Models and Applications
http://dx.doi.org/10.5772/intechopen.94615
Edited by Jaydip Sen
Assistant to the Editor: Sidra Mehtab
Contributors
Pooja Kherwa, Saheel Ahmed, Pranay Berry, Sahil K


In [19]:
#using regex to find section headers
import re

SECTION_PATTERNS = [
    r'(?:^|\n)\s*((?:Chapter|CHAPTER)\s+\d+(?:\.\d+)*\.?\s*[:\-\.]?\s*.+)',
    r'(?:^|\n)\s*((?:Section|SECTION)\s+\d+(?:\.\d+)*\.?\s*[:\-\.]?\s*.+)',
    r'(?:^|\n)\s*(\d+\.\d+(?:\.\d+)*\.?\s+[A-Z][^\n]{3,80})',
    r'(?:^|\n)\s*(\d+\.\s+[A-Z][^\n]{3,80})',
    r'(?:^|\n)\s*((?:Abstract|Introduction|Background|Methodology|Methods|Results|Discussion|Conclusion|Conclusions|References|Bibliography|Acknowledgements|Related Work)\s*\n)',
]

matches = []
for pattern in SECTION_PATTERNS:
    for m in re.finditer(pattern, full_text):
        header = m.group(1).strip()
        if 3 < len(header) < 120:
            matches.append({"header": header, "start": m.start(1)})

# Sort and deduplicate
matches.sort(key=lambda x: x["start"])
deduped = [matches[0]] if matches else []
for m in matches[1:]:
    if m["start"] - deduped[-1]["start"] > 20:
        deduped.append(m)

print(f" Found {len(deduped)} sections:\n")
for m in deduped[:20]:
    print(f"  [{m['start']:>6}] {m['header'][:80]}")

 Found 241 sections:

  [  6336] Chapter 1 
1
  [  6477] Chapter 2 
15
  [  6601] Chapter 3 
47
  [  6749] Chapter 4 
59
  [  6899] Chapter 5 
79
  [  7157] Chapter 6 
103
  [ 17161] Chapter 1
Introductory Chapter: Machine
  [ 17297] 1. Introduction
  [ 20295] Section 5 discusses some new challenges and barriers currently faced by the fina
  [ 20541] 2. Emerging application of machine learning in finance
  [ 28879] 3. Emerging paradigms in computing in finance
  [ 30407] 4. Emerging trends in modeling techniques
  [ 34818] 5. New challenges in financial modeling
  [ 40568] 6. Emerging risks, new choices, and modern practices
  [ 51609] 7. Conclusion
  [ 53663] References
  [ 55413] 589. DOI: 10.4236/jmf.2019.93029
  [ 59148] 2020. DOI: 10.36227/techrxiv.
  [ 59430] 2019. DOI: 10.36227/
  [ 61059] Chapter 2
Design and Analysis of Robust


In [21]:
#classifying sections into types using a simple keyword-based approach

SECTION_KEYWORDS = {
    "introduction": ["introduction", "overview"],
    "background": ["background", "preliminaries"],
    "related_work": ["related work", "previous work"],
    "methodology": ["methodology", "methods", "approach", "proposed"],
    "results": ["results", "findings", "performance"],
    "discussion": ["discussion", "implications"],
    "conclusion": ["conclusion", "summary", "future work"],
    "references": ["references", "bibliography"],
}

def classify_section(title):
    t = title.lower()
    for sec_type, keywords in SECTION_KEYWORDS.items():
        if any(kw in t for kw in keywords):
            return sec_type
    return "general"

# Build sections with text
sections = []
for i, m in enumerate(deduped):
    end = deduped[i + 1]["start"] if i + 1 < len(deduped) else len(full_text)
    text = full_text[m["start"]:end].strip()
    sec_type = classify_section(m["header"])
    
    # Find page number
    page_num = 1
    for pt in reversed(page_texts):
        if m["start"] >= pt["start"]:
            page_num = pt["page"]
            break
    
    sections.append({
        "title": m["header"][:100],
        "text": text,
        "type": sec_type,
        "page": page_num,
        "start": m["start"]
    })

print(f" Classified {len(sections)} sections:\n")
for s in sections[:15]:
    print(f"  [{s['type']:>15}] Page {s['page']:>3} | {s['title'][:60]}")

 Classified 241 sections:

  [        general] Page  15 | Chapter 1 
1
  [        general] Page  15 | Chapter 2 
15
  [        general] Page  15 | Chapter 3 
47
  [        general] Page  15 | Chapter 4 
59
  [        general] Page  15 | Chapter 5 
79
  [        general] Page  15 | Chapter 6 
103
  [        general] Page  21 | Chapter 1
Introductory Chapter: Machine
  [   introduction] Page  21 | 1. Introduction
  [        general] Page  22 | Section 5 discusses some new challenges and barriers current
  [        general] Page  22 | 2. Emerging application of machine learning in finance
  [        general] Page  24 | 3. Emerging paradigms in computing in finance
  [        general] Page  24 | 4. Emerging trends in modeling techniques
  [        general] Page  25 | 5. New challenges in financial modeling
  [        general] Page  27 | 6. Emerging risks, new choices, and modern practices
  [     conclusion] Page  29 | 7. Conclusion


In [22]:
#chunk sections into smaller pieces for embedding

import uuid

CHUNK_SIZE = 512  
CHUNK_OVERLAP = 50
MIN_CHUNK_SIZE = 50

def estimate_tokens(text):
    return len(text) // 4

def split_at_sentences(text, max_tokens, overlap):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current, current_tokens = [], [], 0
    
    for sent in sentences:
        sent_tokens = estimate_tokens(sent)
        if current_tokens + sent_tokens > max_tokens and current:
            chunks.append(" ".join(current))

            # Overlap
            overlap_sents, overlap_tokens = [], 0
            for s in reversed(current):
                t = estimate_tokens(s)
                if overlap_tokens + t > overlap:
                    break
                overlap_sents.insert(0, s)
                overlap_tokens += t
            current = overlap_sents
            current_tokens = overlap_tokens
        current.append(sent)
        current_tokens += sent_tokens
    
    if current:
        chunks.append(" ".join(current))
    return chunks

doc_id = str(uuid.uuid4())[:8]
all_chunks = []

for i, sec in enumerate(sections):
    text = sec["text"].strip()
    if len(text) < MIN_CHUNK_SIZE:
        continue
    
    if estimate_tokens(text) <= CHUNK_SIZE:
        all_chunks.append({
            "id": f"{doc_id}_s{i}_{len(all_chunks)}",
            "text": text,
            "metadata": {
                "chunk_id": f"{doc_id}_s{i}_{len(all_chunks)}",
                "section_title": sec["title"],
                "section_type": sec["type"],
                "page_number": sec["page"],
                "section_number": str(i + 1),
                "doc_id": doc_id
            }
        })
    else:
        sub_chunks = split_at_sentences(text, CHUNK_SIZE, CHUNK_OVERLAP)
        for sub in sub_chunks:
            if len(sub.strip()) < MIN_CHUNK_SIZE:
                continue
            all_chunks.append({
                "id": f"{doc_id}_s{i}_{len(all_chunks)}",
                "text": sub.strip(),
                "metadata": {
                    "chunk_id": f"{doc_id}_s{i}_{len(all_chunks)}",
                    "section_title": sec["title"],
                    "section_type": sec["type"],
                    "page_number": sec["page"],
                    "section_number": str(i + 1),
                    "doc_id": doc_id
                }
            })

print(f" Created {len(all_chunks)} chunks from {len(sections)} sections\n")
for c in all_chunks[:5]:
    print(f"  ID: {c['id']}")
    print(f"  Type: {c['metadata']['section_type']} | Page: {c['metadata']['page_number']}")
    print(f"  Text: {c['text'][:100]}...\n")


 Created 318 chunks from 241 sections

  ID: 245b7c24_s0_0
  Type: general | Page: 15
  Text: Chapter 1 
1
Introductory Chapter: Machine Learning in Finance-Emerging 
Trends and Challenges
by Ja...

  ID: 245b7c24_s1_1
  Type: general | Page: 15
  Text: Chapter 2 
15
Design and Analysis of Robust Deep Learning Models for Stock 
Price Prediction
by Jayd...

  ID: 245b7c24_s2_2
  Type: general | Page: 15
  Text: Chapter 3 
47
Articulated Human Pose Estimation Using Greedy Approach
by Pooja Kherwa, Sonali Singh,...

  ID: 245b7c24_s3_3
  Type: general | Page: 15
  Text: Chapter 4 
59
Ensemble Machine Learning Algorithms for Prediction 
and Classification of Medical Ima...

  ID: 245b7c24_s4_4
  Type: general | Page: 15
  Text: Chapter 5 
79
Delivering Precision Medicine to Patients with Spinal Cord Disorders; 
Insights into A...



In [23]:
#initializing embeddings
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embed_model = SentenceTransformer(EMBEDDING_MODEL)

# Test single embedding
test_emb = embed_model.encode("deep learning for stock prediction")
print(f" Embedding model loaded: {EMBEDDING_MODEL}")
print(f"   Dimensions: {len(test_emb)}")
print(f"   First 5: {test_emb[:5]}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1287.66it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
   Dimensions: 384
   First 5: [-0.09672454 -0.10018361 -0.01886287  0.03830072 -0.07252221]


In [24]:
#initialize chromadb and index chunks
import chromadb

CHROMA_DIR = "./chroma_db"
client = chromadb.PersistentClient(path=CHROMA_DIR)

# Reset collection
try:
    client.delete_collection("documents")
except:
    pass

collection = client.get_or_create_collection(
    name="documents",
    metadata={"hnsw:space": "cosine"}
)

In [25]:
# Embed and index all chunks
texts = [c["text"] for c in all_chunks]
ids = [c["id"] for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]
embeddings = embed_model.encode(texts).tolist()

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas
)

print(f"yes Indexed {collection.count()} chunks into ChromaDB")


yes Indexed 318 chunks into ChromaDB


In [30]:

#testing naive rag retrieval

from langchain_core.prompts import PromptTemplate

def basic_search(query, k=5):
    q_emb = embed_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    docs = []
    for i in range(len(results["ids"][0])):
        docs.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "score": 1 - results["distances"][0][i]
        })
    return docs

general_rag_prompt = PromptTemplate(
    input_variables=["question"],
    template="""You are a machine learning expert. Given a technical question,
write a short passage (2-3 sentences) that would appear in an ML textbook.
Include specific terms like model architectures, algorithms, and metrics.

Question: {question}

Passage:"""
)

query = "What methodology was used in this study?"
results = basic_search(query)

print(f"🔍 Basic RAG: '{query}'\n")
for i, r in enumerate(results):
    print(f"  [{i+1}] Score: {r['score']:.4f} | {r['metadata']['section_type']} | Page {r['metadata']['page_number']}")
    print(f"      {r['text'][:120]}...\n")

🔍 Basic RAG: 'What methodology was used in this study?'

  [1] Score: 0.3984 | general | Page 134
      9. Life case studies of the use of PAAs in institutions
In an attempt to simplify the conceptual complexities of PAAs, a...

  [2] Score: 0.3827 | general | Page 125
      Maria Wawer (my boss) and other colleagues were presenting 
papers on the project findings. During that first week of th...

  [3] Score: 0.3706 | general | Page 137
      9.12 Intervention strategy
The intervention strategy that has been recommended is the use of an ML 
algorithm that calcu...

  [4] Score: 0.3623 | general | Page 134
      A 
particular level of significance is used to determine whether there is a likelihood 
of heart failure. The input vari...

  [5] Score: 0.3588 | general | Page 132
      8. Stages of PAA development
This section explains a more streamlined and contextual version of cross indus-
try standar...



In [31]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7, max_tokens=300)

NUM_HYPOTHESES = 3  # Generate N hypothetical passages

# Two prompts: general academic + technical ML
general_hyde_prompt = PromptTemplate(
    input_variables=["question"],
    template="""You are a research document analyst. Given a question about an
academic document, write a short passage (2-3 sentences) that would appear
in a research paper addressing this topic. Use formal academic language.

Question: {question}

Passage:"""
)

technical_hyde_prompt = PromptTemplate(
    input_variables=["question"],
    template="""You are a machine learning expert. Given a technical question,
write a short passage (2-3 sentences) that would appear in an ML textbook.
Include specific terms like model architectures, algorithms, and metrics.

Question: {question}

Passage:"""
)

general_chain = general_hyde_prompt | llm | StrOutputParser()
technical_chain = technical_hyde_prompt | llm | StrOutputParser()

query = "How does deep learning apply to stock prediction?"
is_technical = True  # Router would decide this

# Step 1: Generate N hypotheses
chain = technical_chain if is_technical else general_chain
hypotheses = []
for i in range(NUM_HYPOTHESES):
    try:
        h = chain.invoke({"question": query}).strip()
        if h:
            hypotheses.append(h)
    except Exception as e:
        print(f"caution Hypothesis {i+1} failed: {e}")

print(f" Generated {len(hypotheses)} hypotheses:\n")
for i, h in enumerate(hypotheses, 1):
    print(f"  [{i}] {h[:120]}...\n")

# Step 2: Search with original query
orig_results = basic_search(query)

# Step 3: Search with EACH hypothesis
all_hyde_results = []
for h in hypotheses:
    hyp_emb = embed_model.encode(h).tolist()
    hyp_results = collection.query(
        query_embeddings=[hyp_emb],
        n_results=5,
        include=["documents", "metadatas", "distances"]
    )
    for j in range(len(hyp_results["ids"][0])):
        all_hyde_results.append({
            "text": hyp_results["documents"][0][j],
            "metadata": hyp_results["metadatas"][0][j],
            "score": 1 - hyp_results["distances"][0][j]
        })

# Step 4: Merge ALL results (original + all hypotheses) and deduplicate
seen = {}
for r in orig_results + all_hyde_results:
    cid = r["metadata"]["chunk_id"]
    if cid not in seen or r["score"] > seen[cid]["score"]:
        seen[cid] = r

# Step 5: Rank by score, return top-k
merged = sorted(seen.values(), key=lambda x: x["score"], reverse=True)[:5]

print(f"search Multi-HyDE Results ({len(orig_results)} original + {len(all_hyde_results)} from {len(hypotheses)} hypotheses → {len(seen)} unique → top 5):\n")
for i, r in enumerate(merged):
    print(f"  [{i+1}] Score: {r['score']:.4f} | {r['metadata']['section_type']} | Page {r['metadata']['page_number']}")
    print(f"      {r['text'][:120]}...\n")

 Generated 3 hypotheses:

  [1] Deep learning applies to stock prediction by leveraging complex model architectures such as recurrent neural networks (R...

  [2] Deep learning applies to stock prediction through the use of complex model architectures, such as recurrent neural netwo...

  [3] Deep learning applies to stock prediction by leveraging complex model architectures, such as recurrent neural networks (...

search Multi-HyDE Results (5 original + 15 from 3 hypotheses → 9 unique → top 5):

  [1] Score: 0.8162 | general | Page 15
      The book presents six chapters that highlight different architectures, models, 
algorithms, and applications of machine ...

  [2] Score: 0.7891 | conclusion | Page 60
      5. Conclusion
Prediction of future stock prices and price movement patterns is a challenging
task if the stock price tim...

  [3] Score: 0.7665 | general | Page 63
      Technical
Report No: NSHM_KOL_2020_SCA_
DS_1, 2020. DOI: 10.13140/
RG.2.2.14022.22085/2. [24] Mehtab, S., S

In [34]:
# testing query router (add-ons)

CONVERSATIONAL = ["hey", "hello", "hi", "thanks", "bye", "what can you do", "who are you"]
SIMPLE = ["section", "chapter", "page", "show me", "what does", "find the"]
TECHNICAL = ["compare", "explain", "how does", "analyze", "summarize", "methodology", "algorithm", "model", "performance"]

def classify_query(query):
    q = query.lower().strip()
    if len(q.split()) <= 4 and any(kw in q for kw in CONVERSATIONAL):
        return "conversational"
    if any(kw in q for kw in SIMPLE):
        return "simple"
    if any(kw in q for kw in TECHNICAL):
        return "complex"
    return "complex"  # default

test_queries = [
    "hey", "hello there", "thanks",
    "What does Chapter 2 say?", "Show me page 5",
    "Compare CNN and LSTM", "What are the key findings?",
    "How does reinforcement learning work?"
]

print(" Router Test:\n")
for q in test_queries:
    r = classify_query(q)
    print(f"  '{q}' → {r}")



 Router Test:

  'hey' → conversational
  'hello there' → conversational
  'thanks' → conversational
  'What does Chapter 2 say?' → simple
  'Show me page 5' → simple
  'Compare CNN and LSTM' → complex
  'What are the key findings?' → complex
  'How does reinforcement learning work?' → complex


In [36]:
#testing generation with retrieved context

gen_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, max_tokens=1500)

system_prompt = (
    "You are a retrieval-grounded assistant for question answering.\n"
    "Answer ONLY using the provided context. Do NOT use outside knowledge.\n"
    "As a knowledgeable and helpful research assistant, your task is to provide informative answers based on the given context.\n"
    "Style rules:\n"
    "- Use 1–3 short sentences.\n"
    "- Prefer wording closely paraphrased from the context.\n"
    "- Include key terms from the question in your answer.\n"
    "- Do not add extra background beyond what the context supports.\n"
    "- Cite sources using [Section X, Page Y] format.\n\n"
    "Context:\n{context}\n"
)

gen_prompt = PromptTemplate(
    input_variables=["question", "context"],
    template=system_prompt + "\nQuestion: {question}\nAnswer:"
)

gen_chain = gen_prompt | gen_llm | StrOutputParser()

# Build context from retrieved docs
query = "What are the main topics covered in this document?"
docs = basic_search(query)

# Filter low scores
relevant = [d for d in docs if d["score"] >= 0.3]

context_parts = []
for i, d in enumerate(relevant, 1):
    sec = d["metadata"]["section_number"]
    page = d["metadata"]["page_number"]
    stype = d["metadata"]["section_type"]
    context_parts.append(f"[{i}] Section {sec}, Page {page} ({stype})\n{d['text'][:500]}")

context = "\n\n---\n\n".join(context_parts)

answer = gen_chain.invoke({"question": query, "context": context})

print(f"❓ Question: {query}\n")
print(f"💬 Answer:\n{answer}\n")
print(f"📌 Citations from {len(relevant)} chunks:")
for d in relevant:
    print(f"  Section {d['metadata']['section_number']}, Page {d['metadata']['page_number']} — {d['metadata']['section_type']}")

❓ Question: What are the main topics covered in this document?

💬 Answer:
The main topics covered in this document include international development programs using PAAs, life case studies of PAAs in institutions, and the stages of PAA development. It also addresses the application and technical perspectives of data analytics, particularly in predictive data analytics [1, Page 143; 2, Page 134; 4, Page 132].

📌 Citations from 5 chunks:
  Section 222, Page 143 — general
  Section 195, Page 134 — general
  Section 6, Page 15 — general
  Section 184, Page 132 — general
  Section 6, Page 15 — general


In [38]:
#testing out of context query
query_ood = "What is the weather in New York?"
docs_ood = basic_search(query_ood)
relevant_ood = [d for d in docs_ood if d["score"] >= 0.3]

if not relevant_ood:
    print(f"Q: '{query_ood}'")
    print(f"ans: I don't have enough information in the provided documents to answer that accurately.")
    print(f" Citations: 0 ")
else:
    print(f" Got {len(relevant_ood)} results — threshold may need tuning")
    for d in relevant_ood:
        print(f"  Score: {d['score']:.4f} — {d['text'][:80]}")

Q: 'What is the weather in New York?'
ans: I don't have enough information in the provided documents to answer that accurately.
 Citations: 0 


In [42]:
#adding memory to the chain

from pathlib import Path
from datetime import datetime

USER_MEMORY_PATH = Path("USER_MEMORY.md")
COMPANY_MEMORY_PATH = Path("COMPANY_MEMORY.md")

def write_memory(path, summary):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    entry = f"\n- [{timestamp}] {summary}"
    if not path.exists():
        header = "# User Memory\n" if "USER" in str(path) else "# Company Memory\n"
        path.write_text(header, encoding="utf-8")
    with open(path, "a", encoding="utf-8") as f:
        f.write(entry + "\n")

def read_memory(path):
    if path.exists():
        content = path.read_text(encoding="utf-8").strip()
        lines = [l for l in content.split("\n") if l.strip() and not l.startswith("#")]
        return "\n".join(lines)
    return ""

# write test memories
write_memory(USER_MEMORY_PATH, "User is interested in deep learning for stock prediction")
write_memory(COMPANY_MEMORY_PATH, "CNN models are faster but LSTM models have comparable accuracy")

print(" User Memory:")
print(f"   {read_memory(USER_MEMORY_PATH)}\n")
print(" Company Memory:")
print(f"   {read_memory(COMPANY_MEMORY_PATH)}")

 User Memory:
   - [2026-02-18 20:30] User is interested in deep learning for stock prediction
- [2026-02-18 20:30] User is interested in deep learning for stock prediction
- [2026-02-18 20:31] User is interested in deep learning for stock prediction

 Company Memory:
   - [2026-02-18 20:30] CNN models are faster but LSTM models have comparable accuracy
- [2026-02-18 20:30] CNN models are faster but LSTM models have comparable accuracy
- [2026-02-18 20:31] CNN models are faster but LSTM models have comparable accuracy


In [43]:
#test memory read in generation
user_mem = read_memory(USER_MEMORY_PATH)
company_mem = read_memory(COMPANY_MEMORY_PATH)

query = "Which model should I use for stock prediction?"
docs = basic_search(query)
relevant = [d for d in docs if d["score"] >= 0.3]

context_parts = []
for i, d in enumerate(relevant, 1):
    context_parts.append(f"[{i}] Section {d['metadata']['section_number']}, Page {d['metadata']['page_number']}\n{d['text'][:500]}")

context = "\n\n---\n\n".join(context_parts)
if user_mem:
    context += f"\n\nUser context: {user_mem}"

answer = gen_chain.invoke({"question": query, "context": context})
print(f"{query}")
print(f"{answer}")

Which model should I use for stock prediction?
You can use deep learning-based regression models for stock prediction. Specifically, four models are based on variants of CNN architectures, and six are constructed using different LSTM architectures for robust and precise predictions [Section 68, Page 60].


In [50]:
#full pipeline test with all add-ons

def full_pipeline(query):
    """Run the complete RAG pipeline."""
    print(f"\n{'='*60}")
    print(f" Query: {query}")
    print(f"{'='*60}")
    
    # Step 1: route
    route = classify_query(query)
    print(f" Route: {route}")
    
    # Step 2: conversational shortcut
    if route == "conversational":
        print(f" Hey! Upload a PDF and ask me questions about it!")
        return
    
    # Step 3: retrieve
    if route == "simple":
        # Simple retrieval: direct vector search
        docs = basic_search(query)
        print(f" Using basic retrieval")
    else:
        # Complex retrieval: multi-HyDE path
        chain = technical_chain if any(kw in query.lower() for kw in ["model", "algorithm", "neural", "learning", "cnn", "lstm"]) else general_chain
        hypotheses = []
        for _ in range(NUM_HYPOTHESES):
            try:
                h = chain.invoke({"question": query}).strip()
                if h:
                    hypotheses.append(h)
            except:
                pass
        print(f" Generated {len(hypotheses)} hypotheses")
        
        # Step 3a: Search with original query
        orig = basic_search(query)
        
        # Step 3b: Search with each hypothesis
        all_hyde = []
        for h in hypotheses:
            hyp_emb = embed_model.encode(h).tolist()
            hyp_res = collection.query(query_embeddings=[hyp_emb], n_results=5, include=["documents", "metadatas", "distances"])
            for j in range(len(hyp_res["ids"][0])):
                all_hyde.append({
                    "text": hyp_res["documents"][0][j],
                    "metadata": hyp_res["metadatas"][0][j],
                    "score": 1 - hyp_res["distances"][0][j]
                })
        
        # Step 3c: Merge + deduplicate
        seen = {}
        for r in orig + all_hyde:
            cid = r["metadata"]["chunk_id"]
            if cid not in seen or r["score"] > seen[cid]["score"]:
                seen[cid] = r

        # Step 3d: Rank by score, return top-k
        merged = sorted(seen.values(), key=lambda x: x["score"], reverse=True)[:5]
        docs = merged

    # Step 4: Filter low scores
    relevant = [d for d in docs if d["score"] >= 0.3]
    print(f" Retrieved: {len(docs)} | Relevant: {len(relevant)}")
    
    if not relevant:
        print(f" I don't have enough information in the provided documents to answer that accurately.")
        return
    
    # Step 5: Build context and generate
    ctx = "\n\n---\n\n".join(
        f"[{i}] Section {d['metadata']['section_number']}, Page {d['metadata']['page_number']} ({d['metadata']['section_type']})\n{d['text'][:500]}"
        for i, d in enumerate(relevant, 1)
    )
    
    user_mem = read_memory(USER_MEMORY_PATH)
    if user_mem:
        ctx += f"\n\nUser context: {user_mem}"
    
    answer = gen_chain.invoke({"question": query, "context": ctx})
    
    print(f"\n Answer:\n{answer}\n")
    print(f" Citations:")
    for d in relevant:
        print(f"  Section {d['metadata']['section_number']}, Page {d['metadata']['page_number']} — {d['metadata']['section_type']} (score: {d['score']:.3f})")

# run tests
full_pipeline("hey")
full_pipeline("What are the main topics covered?")
full_pipeline("Compare CNN and LSTM models for stock prediction")
full_pipeline("What does Chapter 1 discuss?")
full_pipeline("What is the weather today?")


 Query: hey
 Route: conversational
 Hey! Upload a PDF and ask me questions about it!

 Query: What are the main topics covered?
 Route: complex
 Generated 3 hypotheses
 Retrieved: 5 | Relevant: 5

 Answer:
The main topics covered include international development programs using PAAs, life case studies of PAAs in institutions, and the stages of PAA development. Additionally, there is a focus on the balanced scorecard (BSC) for aligning business goals and resources [1, Page 143; 2, Page 134; 4, Page 127; 5, Page 132].

 Citations:
  Section 222, Page 143 — general (score: 0.530)
  Section 195, Page 134 — general (score: 0.435)
  Section 240, Page 148 — conclusion (score: 0.360)
  Section 172, Page 127 — general (score: 0.353)
  Section 184, Page 132 — general (score: 0.351)

 Query: Compare CNN and LSTM models for stock prediction
 Route: complex
 Generated 3 hypotheses
 Retrieved: 5 | Relevant: 5

 Answer:
CNN and LSTM models are both used for stock prediction, with four models based o

In [51]:
#check sanity (sanity_outpyut.json)

import json

results = {
    "status": "running",
    "features": {"rag": False, "memory": False, "citations": False},
    "steps": [],
    "errors": []
}

# Test RAG
try:
    docs = basic_search("What are the main findings?")
    relevant = [d for d in docs if d["score"] >= 0.3]
    if relevant:
        ctx = "\n".join(d["text"][:300] for d in relevant[:3])
        answer = gen_chain.invoke({"question": "What are the main findings?", "context": ctx})
        if len(answer) > 10:
            results["features"]["rag"] = True
        if relevant:
            results["features"]["citations"] = True
        results["steps"].append({"query": "What are the main findings?", "answer_length": len(answer), "citations": len(relevant)})
except Exception as e:
    results["errors"].append(str(e))

# Test Memory
user_mem = read_memory(USER_MEMORY_PATH)
company_mem = read_memory(COMPANY_MEMORY_PATH)
if user_mem or company_mem:
    results["features"]["memory"] = True
results["steps"].append({"user_memory": bool(user_mem), "company_memory": bool(company_mem)})

results["status"] = "passed" if all(results["features"].values()) else "partial"

# Write
os.makedirs("artifacts", exist_ok=True)
with open("artifacts/sanity_output.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\n Status: {results['status']}")
print(f"   Features: {results['features']}")
print(f"   Output: artifacts/sanity_output.json")


 Status: passed
   Features: {'rag': True, 'memory': True, 'citations': True}
   Output: artifacts/sanity_output.json


In [56]:
#test open-meteo api (out-of-context retrieval test)
import requests
import json

# Open-Meteo API — NO API key needed
BASE_URL = "https://api.open-meteo.com/v1/forecast"

# Test: Get 7-day weather for New York
params = {
    "latitude": 40.71,
    "longitude": -74.01,
    "hourly": "temperature_2m,wind_speed_10m,relative_humidity_2m",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "America/New_York",
    "past_days": 7,
    "forecast_days": 7
}

response = requests.get(BASE_URL, params=params)
data = response.json()

print(f"API Response: {response.status_code}")
print(f" Location: {data.get('latitude')}, {data.get('longitude')}")
print(f" Timezone: {data.get('timezone')}")
print(f" Hourly data points: {len(data['hourly']['time'])}")
print(f" Daily data points: {len(data['daily']['time'])}")
print(f"\nDaily temps (max): {data['daily']['temperature_2m_max'][:5]}...")
print(f"Daily temps (min): {data['daily']['temperature_2m_min'][:5]}...")

API Response: 200
 Location: 40.710335, -73.99308
 Timezone: America/New_York
 Hourly data points: 336
 Daily data points: 14

Daily temps (max): [2.6, 3.9, 8.3, 4.4, 3.9]...
Daily temps (min): [-2.7, -5.1, -1.3, -0.6, -1.7]...


In [57]:
#parse time data series
import numpy as np

# Extract daily data
daily = data["daily"]
dates = daily["time"]
temp_max = np.array([t if t is not None else np.nan for t in daily["temperature_2m_max"]])
temp_min = np.array([t if t is not None else np.nan for t in daily["temperature_2m_min"]])
precip = np.array([p if p is not None else np.nan for p in daily["precipitation_sum"]])

# Extract hourly data
hourly = data["hourly"]
h_times = hourly["time"]
h_temp = np.array([t if t is not None else np.nan for t in hourly["temperature_2m"]])
h_wind = np.array([w if w is not None else np.nan for w in hourly["wind_speed_10m"]])
h_humidity = np.array([h if h is not None else np.nan for h in hourly["relative_humidity_2m"]])

print(f" Date range: {dates[0]} to {dates[-1]}")
print(f" Temp max range: {np.nanmin(temp_max):.1f}°C to {np.nanmax(temp_max):.1f}°C")
print(f" Temp min range: {np.nanmin(temp_min):.1f}°C to {np.nanmax(temp_min):.1f}°C")
print(f" Total precipitation: {np.nansum(precip):.1f}mm")
print(f" Hourly data points: {len(h_temp)}")

 Date range: 2026-02-12 to 2026-02-25
 Temp max range: 1.7°C to 8.3°C
 Temp min range: -5.1°C to 1.8°C
 Total precipitation: 30.2mm
 Hourly data points: 336


In [59]:
#compute analytics
def compute_analytics(temps, winds, humidity, dates_h):
    """
    Compute basic time series analytics.
    This runs in our "sandbox" — demonstrating safe computation.
    """
    analytics = {}

    # --- Temperature Analytics ---
    analytics["temperature"] = {
        "mean": round(float(np.nanmean(temps)), 2),
        "std": round(float(np.nanstd(temps)), 2),
        "min": round(float(np.nanmin(temps)), 2),
        "max": round(float(np.nanmax(temps)), 2),
        "range": round(float(np.nanmax(temps) - np.nanmin(temps)), 2),
    }

    # Rolling average (24h window)
    window = 24
    if len(temps) >= window:
        rolling_avg = np.convolve(np.nan_to_num(temps), np.ones(window)/window, mode='valid')
        analytics["temperature"]["rolling_avg_24h"] = {
            "first": round(float(rolling_avg[0]), 2),
            "last": round(float(rolling_avg[-1]), 2),
            "trend": "rising" if rolling_avg[-1] > rolling_avg[0] else "falling"
        }

    # --- Wind Analytics ---
    analytics["wind"] = {
        "mean": round(float(np.nanmean(winds)), 2),
        "max": round(float(np.nanmax(winds)), 2),
        "calm_hours": int(np.sum(winds < 5)),  # below 5 km/h = calm
        "gusty_hours": int(np.sum(winds > 30)),  # above 30 km/h = gusty
    }

    # --- Humidity Analytics ---
    analytics["humidity"] = {
        "mean": round(float(np.nanmean(humidity)), 2),
        "min": round(float(np.nanmin(humidity)), 2),
        "max": round(float(np.nanmax(humidity)), 2),
    }

    # --- Missingness Check ---
    analytics["data_quality"] = {
        "total_points": len(temps),
        "missing_temp": int(np.sum(np.isnan(temps))),
        "missing_wind": int(np.sum(np.isnan(winds))),
        "missing_humidity": int(np.sum(np.isnan(humidity))),
        "completeness_pct": round(
            (1 - np.mean([np.mean(np.isnan(temps)), np.mean(np.isnan(winds)), np.mean(np.isnan(humidity))])) * 100, 1
        )
    }

    # --- Anomaly Detection (simple: beyond 2 std devs) ---
    temp_mean = np.nanmean(temps)
    temp_std = np.nanstd(temps)
    anomaly_mask = np.abs(temps - temp_mean) > (2 * temp_std)
    anomaly_indices = np.where(anomaly_mask)[0]
    analytics["anomalies"] = {
        "count": int(len(anomaly_indices)),
        "threshold": f"±{round(2*temp_std, 1)}°C from mean",
        "timestamps": [dates_h[i] for i in anomaly_indices[:5]] if len(anomaly_indices) > 0 else []
    }

    # --- Volatility (hourly temp changes) ---
    temp_diffs = np.abs(np.diff(np.nan_to_num(temps)))
    analytics["volatility"] = {
        "mean_hourly_change": round(float(np.mean(temp_diffs)), 2),
        "max_hourly_change": round(float(np.max(temp_diffs)), 2),
    }

    return analytics

results = compute_analytics(h_temp, h_wind, h_humidity, h_times)

print("📊 Analytics Results:\n")
print(json.dumps(results, indent=2))

📊 Analytics Results:

{
  "temperature": {
    "mean": 1.46,
    "std": 2.44,
    "min": -5.1,
    "max": 8.3,
    "range": 13.4,
    "rolling_avg_24h": {
      "first": 0.11,
      "last": 1.07,
      "trend": "rising"
    }
  },
  "wind": {
    "mean": 14.79,
    "max": 45.2,
    "calm_hours": 17,
    "gusty_hours": 22
  },
  "humidity": {
    "mean": 75.34,
    "min": 37.0,
    "max": 100.0
  },
  "data_quality": {
    "total_points": 336,
    "missing_temp": 0,
    "missing_wind": 0,
    "missing_humidity": 0,
    "completeness_pct": 100.0
  },
  "anomalies": {
    "count": 19,
    "threshold": "\u00b14.9\u00b0C from mean",
    "timestamps": [
      "2026-02-13T01:00",
      "2026-02-13T02:00",
      "2026-02-13T03:00",
      "2026-02-13T04:00",
      "2026-02-13T05:00"
    ]
  },
  "volatility": {
    "mean_hourly_change": 0.49,
    "max_hourly_change": 2.7
  }
}


In [60]:
#formatting results in a human readable way

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, max_tokens=500)

weather_prompt = PromptTemplate(
    input_variables=["location", "analytics"],
    template="""You are a weather data analyst. Given analytics computed from
Open-Meteo time series data, provide a clear summary of the findings.

Location: {location}
Analytics: {analytics}

Provide a concise summary covering:
1. Temperature overview and trend
2. Wind conditions
3. Any anomalies detected
4. Data quality assessment
Keep it to 4-5 sentences. Be specific with numbers."""
)

weather_chain = weather_prompt | llm | StrOutputParser()

summary = weather_chain.invoke({
    "location": "New York City (40.71°N, 74.01°W)",
    "analytics": json.dumps(results, indent=2)
})

print(f" Weather Analysis Summary:\n\n{summary}")

 Weather Analysis Summary:

In New York City, the mean temperature is 1.46°C, with a range from -5.1°C to 8.3°C and a standard deviation of 2.44°C, indicating some variability. The 24-hour rolling average shows a rising trend, increasing from 0.11°C to 1.07°C. Wind conditions average 14.79 km/h, with gusts reaching up to 45.2 km/h, and there were 22 hours classified as gusty. A total of 19 anomalies were detected, with significant deviations exceeding ±4.9°C from the mean, particularly noted on February 13, 2026. Data quality is excellent, with 336 total points and 100% completeness, showing no missing values for temperature, wind, or humidity.


In [61]:
#full open-meteo tool 

# geocoding api to convert city name → lat/lon


GEO_URL = "https://geocoding-api.open-meteo.com/v1/search"

def geocode_city(city_name):
    """Convert city name to lat/lon using Open-Meteo geocoding."""
    res = requests.get(GEO_URL, params={"name": city_name, "count": 1})
    data = res.json()
    if "results" in data and len(data["results"]) > 0:
        r = data["results"][0]
        return {
            "name": r["name"],
            "country": r.get("country", ""),
            "latitude": r["latitude"],
            "longitude": r["longitude"],
            "timezone": r.get("timezone", "UTC")
        }
    return None

def fetch_weather(latitude, longitude, timezone="UTC", past_days=7, forecast_days=7):
    """Fetch weather data from Open-Meteo API."""
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m,wind_speed_10m,relative_humidity_2m",
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": timezone,
        "past_days": past_days,
        "forecast_days": forecast_days
    }
    res = requests.get(BASE_URL, params=params)
    return res.json()

def analyze_weather(city_name):
    """Full pipeline: geocode → fetch → analyze → summarize."""
    # Geocode
    geo = geocode_city(city_name)
    if not geo:
        return f"Could not find location: {city_name}"

    print(f"📍 Found: {geo['name']}, {geo['country']} ({geo['latitude']}, {geo['longitude']})")

    # Fetch
    data = fetch_weather(geo["latitude"], geo["longitude"], geo["timezone"])

    # Parse
    hourly = data["hourly"]
    temps = np.array([t if t is not None else np.nan for t in hourly["temperature_2m"]])
    winds = np.array([w if w is not None else np.nan for w in hourly["wind_speed_10m"]])
    humid = np.array([h if h is not None else np.nan for h in hourly["relative_humidity_2m"]])
    times = hourly["time"]

    # Analyze
    analytics = compute_analytics(temps, winds, humid, times)

    # Summarize
    summary = weather_chain.invoke({
        "location": f"{geo['name']}, {geo['country']} ({geo['latitude']}°N, {geo['longitude']}°W)",
        "analytics": json.dumps(analytics, indent=2)
    })

    return {
        "location": geo,
        "analytics": analytics,
        "summary": summary
    }

# Test with different cities
for city in ["Tokyo", "London", "Mumbai"]:
    print(f"\n{'='*50}")
    result = analyze_weather(city)
    if isinstance(result, dict):
        print(f"\n {result['summary']}")
    else:
        print(result)


📍 Found: Tokyo, Japan (35.6895, 139.69171)

 In Tokyo, Japan, the average temperature is 8.02°C, with a standard deviation of 4.61°C, ranging from a minimum of -0.5°C to a maximum of 20.1°C. The 24-hour rolling average shows a rising trend, increasing from 5.57°C to 12.69°C. Wind conditions are relatively calm, with a mean speed of 5.96 km/h and no recorded gusty hours, although there were 194 calm hours. Eight anomalies were detected, all exceeding the threshold of ±9.2°C from the mean, occurring on February 23, 2026, during the hours from 10:00 to 14:00. Data quality is excellent, with 336 total data points and 100% completeness, indicating no missing values for temperature, wind, or humidity.

📍 Found: London, United Kingdom (51.50853, -0.12574)

 In London, the average temperature is 8.14°C, with a range from a minimum of 1.3°C to a maximum of 15.2°C, and a rising trend in the 24-hour rolling average from 8.62°C to 11.95°C. Wind conditions show a mean speed of 13.51 km/h, with a m

In [62]:

WEATHER_KEYWORDS = [
    "weather", "temperature", "forecast", "climate",
    "rain", "wind", "humidity", "snow", "storm",
    "hot", "cold", "warm", "celsius", "fahrenheit",
    "open-meteo", "open meteo"
]

LOCATION_PATTERNS = [
    r"weather (?:in|for|at) (.+?)(?:\?|$|\.)",
    r"temperature (?:in|for|at) (.+?)(?:\?|$|\.)",
    r"forecast (?:in|for|at) (.+?)(?:\?|$|\.)",
    r"how (?:hot|cold|warm) is (?:it in )?(.+?)(?:\?|$|\.)",
]

import re

def detect_weather_query(query):
    """Check if query is a weather/tool-call query and extract location."""
    q = query.lower().strip()

    if not any(kw in q for kw in WEATHER_KEYWORDS):
        return None

    # Try to extract location
    for pattern in LOCATION_PATTERNS:
        m = re.search(pattern, q, re.IGNORECASE)
        if m:
            return m.group(1).strip()

    return "unknown_location"

test_queries = [
    "What's the weather in Paris?",
    "Temperature forecast for Tokyo",
    "How cold is it in Moscow?",
    "Compare CNN and LSTM models",  # should NOT trigger
    "What does Chapter 2 say?",     # should NOT trigger
    "hey",                          # should NOT trigger
    "Show me weather data for London",
]

print("🌤️ Weather Query Detection:\n")
for q in test_queries:
    result = detect_weather_query(q)
    if result:
        print(f"   '{q}' → location: '{result}'")
    else:
        print(f"   '{q}' → not a weather query")

🌤️ Weather Query Detection:

   'What's the weather in Paris?' → location: 'paris'
   'Temperature forecast for Tokyo' → location: 'tokyo'
   'How cold is it in Moscow?' → location: 'moscow'
   'Compare CNN and LSTM models' → not a weather query
   'What does Chapter 2 say?' → not a weather query
   'hey' → not a weather query
   'Show me weather data for London' → location: 'unknown_location'
